# Generating the Solutions for the Selected Problems
Here, I want to use **Qwen2.5-Coder-14B-Instruct** to generate the solutions for the selected problems. Each of them will be written in Python, C#, and Typescript. The model is available on [Hugging Face](https://huggingface.co/Qwen/Qwen2.5-Coder-14B-Instruct).

In [2]:
# !pip install transformers
# !pip install collections 

In [3]:
import re
import os
import math
import torch
import pandas as pd
from tqdm import tqdm
from enum import Enum
from collections import defaultdict
from transformers import AutoTokenizer, AutoModelForCausalLM

## Selected Problems Loading
Firstly, I need to load the selected problems dataframe.

In [4]:
csv_file = '/datasets/leetcode_selected_problems.csv'
csv_full_path = os.getcwd()+csv_file
if os.path.exists(csv_full_path):
    selected_problems = pd.read_csv(csv_full_path)
else:
    selected_problems = None
    print("File not found.")

In [5]:
selected_problems["solution"] = ''

In [6]:
selected_problems.head()

,title,description,difficulty,solution
0,Relative Ranks,You are given an integer array `score` of size...,Easy,
1,Truncate Sentence,A sentence is a list of words that are separat...,Easy,
2,Perform String Shifts,You are given a string `s` containing lowercas...,Easy,
3,Invert Binary Tree,"Given the `root` of a binary tree, invert the ...",Easy,
4,Merge Strings Alternately,You are given two strings `word1` and `word2`....,Easy,


## Loading Qwen2.5-Coder-14B-Instruct Model

In [7]:
class Language(Enum):
    PYTHON = "python"
    TYPESCRIPT = "typescript"
    C_SHARP = "csharp"

In [8]:
model_name = "Qwen/Qwen2.5-Coder-14B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

2025-04-02 15:29:39.654223: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-02 15:29:39.694388: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-04-02 15:29:39.694419: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-04-02 15:29:39.694451: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-02 15:29:39.702727: I tensorflow/core/platform/cpu_feature_g

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

In [9]:
c_sharp_prompt = "You are a C# developer. Please, solve this problem avoid using any built-in functions or libraries that simplify algorithmic steps:\n"
python_prompt = "You are a Python developer. Please, solve this problem avoid using any built-in functions or libraries that simplify algorithmic steps:\n"
typescript_prompt = "You are a Typescript developer. Please, solve this problem avoid using any built-in functions or libraries that simplify algorithmic steps:\n"

In [94]:
extra_instructions = "\nDo not add any extra text, explanations or comments."

In [10]:
def get_code_from_text(text, programming_language):
    code_match = re.search(r'```(.*?)```', text, re.DOTALL)
    if code_match:
        extracted_code = code_match.group(1).strip().replace(programming_language.value+'\n', '')
        return extracted_code
    else:
        return "No code found in the generated output"

In [86]:
def get_normalized_prob_line_by_line(logits, generated_tokens, context_length, generated_code, comment_symbol, programming_language):
    token_probs = defaultdict(list)
    
    code_token = "``"
    code_started = False
    
    current_line = 0
    lines = defaultdict(list)
    current_line_text = ""
    comment_started = False

    for i, token_logits in enumerate(logits): 
        token_id = generated_tokens[i + context_length].item()
        decoded_token = tokenizer.decode(token_id, skip_special_tokens=True)
        current_line_text += decoded_token
        if not code_started and code_token not in decoded_token: # the code not started
            continue

        if not code_started and code_token in decoded_token: # the code started
            code_started = True
            continue
        
        if code_started and code_token in decoded_token: # the code finished
            break
        
        if comment_symbol in decoded_token:
            comment_started = True
        
        if not comment_started:
            token_prob = torch.softmax(token_logits, dim=-1)[0][token_id].item()
            token_probs[current_line].append(token_prob)

        if '\n' in decoded_token:
            lines[current_line] = current_line_text
            current_line_text = ""
            current_line += 1
            token_probs[current_line] = []
            comment_started = False

    
    probs = defaultdict(list)
    for key, value in token_probs.items():
        L = len(token_probs[key])
        if L > 0:
            normalized_prob = math.prod([p ** (1 / L) for p in value])
            probs[key] = normalized_prob
            # print(f"Line [{key}]: {lines[key]} \n {normalized_prob}")
    return lines, probs

In [89]:
max_context_length = model.config.max_position_embeddings

def get_generated_code(prompt, programming_language, comment_symbol):
    inputs = tokenizer(prompt, return_tensors="pt", padding=True)
    input_ids = inputs['input_ids']
    context_length = input_ids.shape[1]
    if context_length >= max_context_length:
        print(f"The prompt is too long: {context_length}; Maximum: {max_context_length}")
        return None

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id
    outputs = model.generate(**inputs, 
                             max_length= max_context_length-1, 
                             output_scores=True,
                             output_logits = True,
                             return_dict_in_generate=True)
    generated_tokens = outputs.sequences[0]
    generated_answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    
    logits = outputs.logits
    generated_code = get_code_from_text(generated_answer, programming_language).strip()
    return generated_code, get_normalized_prob_line_by_line(logits, generated_tokens, context_length, generated_code, comment_symbol, programming_language)

In [93]:
easy_test = get_generated_code("Can you write a Python function that computes the factorial of a number? Do not add any extra text, explanations or comments.", Language.PYTHON, '#')
print(easy_test[0])
print()
print(easy_test[1][0])
print(easy_test[1][1])

def factorial(n):
    if n == 0:
        return 1
    else:
        return n * factorial(n-1)

defaultdict(<class 'list'>, {0: ' ```python\n', 1: 'def factorial(n):\n', 2: '    if n == 0:\n', 3: '        return 1\n', 4: '    else:\n', 5: '        return n * factorial(n-1)\n'})
defaultdict(<class 'list'>, {0: 0.9833444678383888, 1: 0.9534250353141046, 2: 0.9555523881129241, 3: 0.9962464336804056, 4: 0.9971034987665034, 5: 0.9501637108676124})


In [95]:
generated_code = get_generated_code(python_prompt+selected_problems.iloc[0]["description"]+extra_instructions, Language.PYTHON, '#')
print(generated_code[0])
print()
print(generated_code[1][0])
print(generated_code[1][1])

def findRelativeRanks(score):
    n = len(score)
    sorted_scores = [None] * n
    answer = [None] * n
    
    # Copy the score array to sort_scores
    for i in range(n):
        sorted_scores[i] = score[i]
    
    # Sort the sorted_scores array
    for i in range(n):
        for j in range(i + 1, n):
            if sorted_scores[i] < sorted_scores[j]:
                temp = sorted_scores[i]
                sorted_scores[i] = sorted_scores[j]
                sorted_scores[j] = temp
    
    # Assign ranks based on sorted order
    for i in range(n):
        for j in range(n):
            if score[i] == sorted_scores[j]:
                if j == 0:
                    answer[i] = "Gold Medal"
                elif j == 1:
                    answer[i] = "Silver Medal"
                elif j == 2:
                    answer[i] = "Bronze Medal"
                else:
                    answer[i] = str(j + 1)
                break
    
    return answer

defaultdict(<class 'list'>, {0: '

## Code Generation

In [96]:
def add_token_prob(title, lines, probs, df):
    for key, value in probs.items():
        df.loc[len(df)] = [title, key, lines[key], value]
    

In [97]:
columns = ['title', 'line_number', 'line_text', 'token_prob']
python_token_prob = pd.DataFrame([], columns=columns)
c_harp_token_prob = pd.DataFrame([], columns=columns)
typescript_token_prob = pd.DataFrame([], columns=columns)

In [103]:
def generate_solution(start_prompt, programming_language, prob_df, comment_symbol):
    problems_with_solutions = selected_problems
    for index, row in tqdm(problems_with_solutions.iterrows()):
        code, linesprobs = get_generated_code(start_prompt+row["description"], programming_language, comment_symbol)
        problems_with_solutions.loc[index, "solution"] = code
        add_token_prob(row["title"], linesprobs[0], linesprobs[1], prob_df)
    return problems_with_solutions

In [108]:
c_sharp_solutions = generate_solution(c_sharp_prompt, Language.C_SHARP, c_harp_token_prob, '//')
c_sharp_solutions.to_csv('datasets/leetcode_selected_problems_c_sharp_14B.csv', index=False)

30it [3:48:39, 457.31s/it]


In [109]:
typescript_solutions = generate_solution(typescript_prompt, Language.TYPESCRIPT, typescript_token_prob, '//')
typescript_solutions.to_csv('datasets/leetcode_selected_problems_typescript_14B.csv', index=False)

30it [3:37:29, 434.97s/it]


In [104]:
python_solutions = generate_solution(python_prompt, Language.PYTHON, python_token_prob, '#')
python_solutions.to_csv('datasets/leetcode_selected_problems_python_14B.csv', index=False)

30it [3:56:44, 473.48s/it]


In [105]:
python_token_prob.to_csv('datasets/probabilities_python.csv', index=False)

In [110]:
c_harp_token_prob.to_csv('datasets/probabilities_c_sharp.csv', index=False)
typescript_token_prob.to_csv('datasets/probabilities_typescript.csv', index=False)

# Delete Comments from Generated Code

In [111]:
python_csv_file = '/datasets/leetcode_selected_problems_python_14B.csv'
c_sharp_csv_file = '/datasets/leetcode_selected_problems_c_sharp_14B.csv'
typescript_csv_file = '/datasets/leetcode_selected_problems_typescript_14B.csv'

def get_generated_code(csv_file):
    csv_full_path = os.getcwd()+csv_file
    if os.path.exists(csv_full_path):
        return pd.read_csv(csv_full_path)
    else:
        print("File not found.")
        return None
    
python_solutions = get_generated_code(python_csv_file)
c_sharp_solutions = get_generated_code(c_sharp_csv_file)
typescript_solutions = get_generated_code(typescript_csv_file)

In [112]:
def remove_comments(code, comment_symbol):
    comment_symbol_escaped = re.escape(comment_symbol)
    code_without_comments = re.sub(f'{comment_symbol_escaped}.*', '', code)
    code_cleaned = re.sub(r'\n\s*\n', '\n', code_without_comments).strip()

    return code_cleaned

code_solution = '''
# Test
def say_hello():  # This function says hello
    print("Hello, world!")  # Print greeting message
'''
cleaned_code = remove_comments(code_solution, '#')
print(cleaned_code)

def say_hello():  
    print("Hello, world!")


In [113]:
python_solutions["solution_no_comments"] = ''
c_sharp_solutions["solution_no_comments"] = ''
typescript_solutions["solution_no_comments"] = ''

In [114]:
def delete_comments(solutions, programming_language):
    comment_symbol = ''
    if programming_language == Language.PYTHON:
        comment_symbol = '#'
    else:
        comment_symbol = '//'

    for index, row in tqdm(solutions.iterrows()):
         solutions.loc[index, 'solution_no_comments'] = remove_comments(row["solution"], comment_symbol)

In [115]:
delete_comments(python_solutions, Language.PYTHON)

30it [00:00, 1619.17it/s]


In [116]:
print(python_solutions[0:1]["solution_no_comments"].to_list()[0])

def findRelativeRanks(score):
    sorted_scores = score[:]
    for i in range(len(sorted_scores)):
        for j in range(i + 1, len(sorted_scores)):
            if sorted_scores[i] < sorted_scores[j]:
                sorted_scores[i], sorted_scores[j] = sorted_scores[j], sorted_scores[i]
    rank_mapping = {}
    for i in range(len(sorted_scores)):
        if i == 0:
            rank_mapping[sorted_scores[i]] = "Gold Medal"
        elif i == 1:
            rank_mapping[sorted_scores[i]] = "Silver Medal"
        elif i == 2:
            rank_mapping[sorted_scores[i]] = "Bronze Medal"
        else:
            rank_mapping[sorted_scores[i]] = str(i + 1)
    answer = []
    for s in score:
        answer.append(rank_mapping[s])
    return answer
print(findRelativeRanks([5, 4, 3, 2, 1]))  
print(findRelativeRanks([10, 3, 8, 9, 4]))


In [117]:
delete_comments(typescript_solutions, Language.TYPESCRIPT)
delete_comments(c_sharp_solutions, Language.C_SHARP)

30it [00:00, 1595.97it/s]
30it [00:00, 2804.99it/s]


In [118]:
c_sharp_solutions.to_csv('datasets/leetcode_selected_problems_c_sharp_14B_no_comments.csv', index=False)
typescript_solutions.to_csv('datasets/leetcode_selected_problems_typescript_14B_no_comments.csv', index=False)
python_solutions.to_csv('datasets/leetcode_selected_problems_python_14B_no_comments.csv', index=False)